# تدريب نموذج محادثة وصال (LoRA / QLoRA عبر Unsloth)

هذا الدفتر يُدرِّب طبقة LoRA خفيفة فوق نموذج مفتوح المصدر (لا فوق Claude أو Kimi أو Gemini —
تلك نماذج مغلقة تُستدعى عبر API فقط ولا يمكن تدريبها بأي شكل). الناتج تخصيص أسلوب/شخصية
الرد، وليس مصدر حقائق موثوقة — راجع القسم الأخير لتفاصيل هذا الفرق ولماذا يهم تحديداً هنا.

**قبل التشغيل:** Runtime ← Change runtime type ← اختر T4 GPU (مجاني) أو أعلى.

**بيانات التدريب:** هذا الدفتر يعمل بمجموعة أمثلة صغيرة مضمَّنة فيه للتحقق من أن كل الأنابيب
تعمل من أول تشغيل. لتدريب حقيقي، صدِّر محادثات وصال الفعلية (مموَّهة الهوية) عبر:
```
php tools/export-chat-training-data.php
```
ثم ارفع الملف الناتج `chat-training-data.jsonl` هنا (الخلية المخصصة لذلك أدناه) بدل الأمثلة
المضمَّنة.

## ١) التثبيت

In [ ]:
%%capture
# إن فشل هذا السطر بخطأ توافق CUDA/torch، راجع أمر التثبيت المحدَّث في:
# https://docs.unsloth.ai (قسم Installation) — Colab يغيّر نسخة torch
# المثبَّتة مسبقاً بين حين وآخر، وUnsloth يحدِّث تعليماته تبعاً لذلك.
!pip install -U unsloth
!pip install -U trl

## ٢) تحميل النموذج الأساس

**تحقّق من `model_name` أدناه قبل التشغيل.** بيئة العمل التي أكتب منها هذا الدفتر محجوبة عن
الوصول لموقع Hugging Face مباشرة، فلا أقدر أؤكّد المعرّف الدقيق للأحدث. حسب آخر بحث متاح لي
(سبتمبر ٢٠٢٦): عائلة **Qwen3** بحجم ٨B تتصدّر النماذج المفتوحة بهذا الحجم على معايير العربية،
ورخصتها Apache 2.0 (تجارية بلا قيود). افتح https://huggingface.co/unsloth من متصفحك،
ابحث عن "Qwen3" واختر نسخة `-Instruct` بحجم ٧-٩B تحمل لاحقة `-bnb-4bit` (أسرع تحميلاً)،
والصق معرّفها الدقيق مكان القيمة أدناه إن اختلف.

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit"  # ⚠️ تحقّق من هذا المعرّف — انظر الملاحظة أعلاه
max_seq_length = 2048
dtype = None        # None = يكتشفها تلقائياً حسب كرت الشاشة
load_in_4bit = True  # ضروري لتشغيل نموذج ٧-٩B على GPU مجاني بذاكرة ~16GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## ٣) إضافة طبقات LoRA

لا نُدرِّب النموذج كاملاً (مليارات المعاملات) — فقط طبقات صغيرة مُضافة فوقه. هذا ما يجعل
التدريب ممكناً على GPU مجاني خلال دقائق بدل أيام.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## ٤) بيانات التدريب

الخلية التالية تبحث عن ملف مرفوع باسم `chat-training-data.jsonl` (اسحبه من شريط الملفات على
اليسار، أو نفّذ `from google.colab import files; files.upload()` في خلية منفصلة). إن لم
تجده، تستخدم مجموعة أمثلة صغيرة مضمَّنة — تكفي للتحقق من أن الأنبوب يعمل، لا لتدريب حقيقي.

In [ ]:
import json, os
from datasets import Dataset

SYSTEM_PROMPT = (
    "أنت مساعد وصال، مساعد ذكاء اصطناعي يخدم الأشخاص ذوي الإعاقة في السعودية. "
    "أجب بوضوح واحترام، وحين يسألك أحد عن حقيقة رسمية (مبلغ، شرط أهلية، إجراء حكومي) "
    "لا تجزم برقم قد يتغيّر — وجّهه للجهة الرسمية المختصة أو مركز الدعم الفني للتأكد."
)

# أمثلة مضمَّنة — استبدلها ببياناتك الحقيقية فور توفّرها
PLACEHOLDER_EXAMPLES = [
    {"instruction": "مين أنت وش تقدر تسوي؟",
     "response": "أنا مساعد وصال، مساعد ذكاء اصطناعي أساعدك في أسئلتك اليومية وأدلّك على خدمات المنصة. إذا احتجت مساعدة تقنية أو دعماً فنياً، أقدر أفتح لك تذكرة دعم مباشرة."},
    {"instruction": "كيف أبلغ عن صعوبة وصول لخدمة إلكترونية؟",
     "response": "من لوحة التحكم ← تذاكر الدعم ← تذكرة جديدة، اختر نوع \"صعوبة وصول\" واشرح المشكلة بالتفصيل (اسم الخدمة، الجهاز، والمتصفح إن أمكن). فريق الدعم يتابعها ويحدّد أولويتها فوراً."},
    {"instruction": "أبي أغيّر بريدي الإلكتروني المسجَّل",
     "response": "من الملف الشخصي ← الإعدادات، تقدر تعدّل بريدك مباشرة. إذا واجهت مشكلة، افتح تذكرة من نوع \"تعديل البريد الإلكتروني\" وفريق الدعم يتابعها معك."},
    {"instruction": "وش الفرق بين الوضع البسيط والمفصّل في المحادثة؟",
     "response": "الوضع البسيط يعطيك إجابة مختصرة ومباشرة، والمفصّل يشرح بخطوات أوضح مع تفاصيل إضافية. تقدر تبدّل بينهما من فوق صندوق المحادثة حسب اللي يناسبك."},
    {"instruction": "أحس أن إجابة المساعد غير دقيقة، أسوي أي شيء؟",
     "response": "بلّغ عنها من خيار \"الإبلاغ عن معلومة غير دقيقة\" في تذاكر الدعم — فريق مختص يراجع الإجابة ويصحّحها، وهذا يساعدنا نحسّن المساعد لجميع المستخدمين."},
    {"instruction": "هل تقدر تعطيني معلومات عن حقوقي كشخص من ذوي الإعاقة في السعودية؟",
     "response": "أقدر أعطيك خطوطاً عامة، لكن التفاصيل الدقيقة (المبالغ، شروط الأهلية) تتغيّر من فترة لأخرى — تأكّد منها دائماً عبر بوابة وزارة الموارد البشرية والتنمية الاجتماعية (hrsd.gov.sa) أو منصة أبشر، حتى تكون المعلومة اللي تعتمد عليها محدَّثة ورسمية."},
    {"instruction": "كيف أتابع تذكرة دعم رفعتها بدون حساب؟",
     "response": "عند إنشاء التذكرة تحصل على رابط تتبّع خاص بها — احتفظ فيه، فهو الطريقة الوحيدة لمتابعة حالتها بدون تسجيل دخول. لو ضاع الرابط، راسل الدعم من نفس البريد اللي سجّلت فيه التذكرة."},
    {"instruction": "هل المنصة تدعم قارئ الشاشة؟",
     "response": "نعم، صُممت الواجهة لتتوافق مع قارئات الشاشة وأحجام الخط الكبيرة، وفيها وضع تباين عالٍ. إذا واجهت أي عائق معها، بلّغنا عبر تذكرة \"صعوبة وصول\" ونعطيها أولوية مباشرة."},
]

if os.path.exists("chat-training-data.jsonl"):
    rows = [json.loads(line) for line in open("chat-training-data.jsonl", encoding="utf-8") if line.strip()]
    print(f"استُخدمت بياناتك الحقيقية: {len(rows)} صفاً")
else:
    rows = PLACEHOLDER_EXAMPLES
    print(f"لا يوجد ملف مرفوع — استُخدمت {len(rows)} مثالاً تجريبياً فقط للتحقق من الأنبوب")

def to_chat_text(example):
    text = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["response"]},
        ],
        tokenize = False,
        add_generation_prompt = False,
    )
    return {"text": text}

dataset = Dataset.from_list(rows).map(to_chat_text)

## ٥) التدريب

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# max_steps=60 يناسب المجموعة التجريبية الصغيرة فقط. لبياناتك الحقيقية،
# احذف max_steps واستخدم num_train_epochs=3 بدلاً منها (أو أكثر حسب حجم البيانات).
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## ٦) اختبار سريع

In [ ]:
FastLanguageModel.for_inference(model)

test_question = "كيف أفتح تذكرة دعم فني؟"
inputs = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test_question},
    ],
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 200, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

## ٧) حفظ النتيجة

In [ ]:
model.save_pretrained("wesal_lora")
tokenizer.save_pretrained("wesal_lora")

# اختياري: رفعه إلى Hugging Face Hub (يحتاج توكن من huggingface.co/settings/tokens)
# model.push_to_hub("اسم_حسابك/wesal-lora", token="hf_...")

from google.colab import files
import shutil
shutil.make_archive("wesal_lora", "zip", "wesal_lora")
files.download("wesal_lora.zip")

## ملاحظة مهمة قبل الاعتماد على هذا في الإنتاج

**Colab ليس استضافة دائمة.** ما تدرّبه هنا ملف أوزان (LoRA adapter) على قرصك — لتشغيله فعلياً
بدل استدعاء Claude/Kimi الحاليين في `api/chat.php`، يحتاج خادم استدلال بمعالج رسومي (GPU) يعمل
طوال الوقت. استضافة وصال الحالية على Hostinger مشتركة ولا تملك GPU إطلاقاً — لن يعمل عليها.
خيارات واقعية: Hugging Face Inference Endpoints، خادم GPU سحابي مخصص (Runpod، Lambda، إلخ)،
أو الإبقاء عليه تجربة بحثية بينما الإنتاج يستمر عبر الـ API الحالي.

**هذا التدريب يُخصّص الأسلوب لا الحقائق.** لا تعتمد عليه لضمان دقّة معلومات رسمية (مبالغ إعانات،
شروط أهلية، إجراءات حكومية) — نموذج مُدرَّب بهذي الطريقة قد "يختلق" رقماً بثقة تامة لو لم يكن في
بيانات تدريبه. للمعلومات الرسمية الدقيقة والقابلة للتحديث، البنية الصحيحة نظام RAG (استرجاع من
قاعدة معرفة موثّقة عند كل سؤال) لا تدريب الأوزان — مشروع منفصل، أخبرني إذا تحب نبدأ فيه.